# WOE Logistic Regression

这是传统评分卡风格的解释性线性基准。WOE、IV、单调分箱、稀有类别、相关性、VIF、P-value 和系数方向全部只在 Train 上学习；Selection 只用于锁模后的开发期评估，Test 保持封存。

In [1]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / 'configs' / 'experiment.yaml').is_file()
)
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from credit_risk.development import load_development_data
from credit_risk.models.woe_logistic import run_woe_logistic
print('Project root:', PROJECT_ROOT)

Project root: /Users/ganlu/Desktop/home-credit-risk-english


In [2]:
development = load_development_data(PROJECT_ROOT)
assert not hasattr(development, 'X_test')
assert not hasattr(development, 'y_test')
display(development.split_overview)
print('公共特征数:', len(development.feature_columns))

,sample,n,bad_rate
0,Train,215257,0.080727
1,Selection,46127,0.080734
2,Test (sealed),46127,NaN


公共特征数: 126


In [3]:
result = run_woe_logistic(development, register_lock=True)
display(result.metrics)
display(result.iv_table.head(30))
display(result.final_parameter_table)
display(result.selection_history)
print('最终评分卡变量数:', len(result.final_features))
print('最终变量:', result.final_features)
print('锁模产物:', result.run_directory)

,sample,n,bad_rate,AUC,AR,KS,AP,Logloss
0,Train,215257,0.080727,0.742165,0.484330,0.360115,0.217048,0.251072
1,Selection,46127,0.080734,0.741919,0.483837,0.355672,0.221226,0.250908


,feature,iv,type,bins,direction,missing_rate,single_ar
0,EXT_SOURCE_3,0.326991,numeric,11,decreasing,0.198275,0.313519
1,EXT_SOURCE_2,0.307641,numeric,11,decreasing,0.002156,0.306048
2,EXT_SOURCE_1,0.153018,numeric,11,decreasing,0.563852,0.177601
3,EMPLOYED_YEARS,0.112593,numeric,10,decreasing,0.180045,0.187387
4,EMPLOYED_AGE_RATIO,0.091479,numeric,10,decreasing,0.180045,0.168070
5,AGE_YEARS,0.086219,numeric,10,decreasing,0.000000,0.166168
6,OCCUPATION_TYPE,0.081734,categorical,15,NaN,0.313486,0.151558
7,GOODS_CREDIT_RATIO,0.068906,numeric,8,decreasing,0.000869,0.138043
8,ORGANIZATION_TYPE,0.066143,categorical,30,NaN,0.000000,0.141975
9,NAME_INCOME_TYPE,0.054586,categorical,5,NaN,0.000000,0.118303


,feature,coefficient,p_value,iv,single_ar,missing_rate,VIF
0,EXT_SOURCE_3,0.835598,0.000000e+00,0.326991,0.313519,0.198275,1.060082
1,EXT_SOURCE_2,0.731142,0.000000e+00,0.307641,0.306048,0.002156,1.203449
2,EXT_SOURCE_1,0.509101,5.209530e-112,0.153018,0.177601,0.563852,1.095358
3,EMPLOYED_YEARS,0.287812,1.071955e-21,0.112593,0.187387,0.180045,1.471004
4,OCCUPATION_TYPE,0.291038,1.593633e-19,0.081734,0.151558,0.313486,1.312790
5,GOODS_CREDIT_RATIO,0.686412,2.148604e-109,0.068906,0.138043,0.000869,1.057911
6,ORGANIZATION_TYPE,0.327482,1.784645e-16,0.066143,0.141975,0.000000,1.524424
7,REGION_RATING_CLIENT_W_CITY,0.299349,1.994080e-14,0.051989,0.098494,0.000000,1.155473
8,NAME_EDUCATION_TYPE,0.516965,4.876662e-35,0.048592,0.091031,0.000000,1.158680
9,PHONE_CHANGE_YEARS,0.177422,2.230161e-05,0.044289,0.110557,0.000005,1.075056


,feature,reason,value
0,NAME_INCOME_TYPE,P_value,0.374120
1,REG_CITY_NOT_WORK_CITY,P_value,0.286681
2,REGION_POPULATION_RELATIVE,P_value,0.092362
3,AGE_YEARS,negative_coefficient,-0.247842
4,AMT_GOODS_PRICE,negative_coefficient,-0.149087


最终评分卡变量数: 18
最终变量: ['EXT_SOURCE_3', 'EXT_SOURCE_2', 'EXT_SOURCE_1', 'EMPLOYED_YEARS', 'OCCUPATION_TYPE', 'GOODS_CREDIT_RATIO', 'ORGANIZATION_TYPE', 'REGION_RATING_CLIENT_W_CITY', 'NAME_EDUCATION_TYPE', 'PHONE_CHANGE_YEARS', 'CREDIT_ANNUITY_RATIO', 'ID_PUBLISH_YEARS', 'FLAG_DOCUMENT_3', 'REGISTRATION_YEARS', 'REG_CITY_NOT_LIVE_CITY', 'NAME_FAMILY_STATUS', 'EMERGENCYSTATE_MODE', 'OWN_CAR_AGE']
锁模产物: /Users/ganlu/Desktop/home-credit-risk-english/artifacts/woe_logistic/20260831T023645_914291Z


In [4]:
registry_path = PROJECT_ROOT / development.configs['experiment']['test_policy']['registry_path']
registry = json.loads(registry_path.read_text(encoding='utf-8'))
assert registry['models']['woe_logistic']['locked'] is True
assert registry['suite_locked'] is False
print('两类 Logistic 均已锁定；非线性模型尚未登记，Test 继续封存。')

两类 Logistic 均已锁定；非线性模型尚未登记，Test 继续封存。
